In [1]:
import duckdb

con = duckdb.connect("../output/investigation.duckdb", read_only=True)


def section(title):
    print()
    print("=" * 90)
    print(title)
    print("=" * 90)


def run(label, sql):
    print(f"\n-- {label} --")
    rows = con.execute(sql).fetchall()
    if not rows:
        print("  (no rows)")
        return
    for r in rows:
        print(r)


In [ ]:
"""
05_bridenstine_verification.py

Fills in all the TODOs from notes/05_finding_bridenstine.md:
- Registrant ID and exact filing count for The Artemis Group
- Specific filing_uuids backing each quantitative claim
- House LDA cross-validation
- Press release corpus scan for Bridenstine mentions and NASA/Artemis rhetoric
"""


# ─────────────────────────────────────────────────────────────────────────────
section("Q7: The Artemis Group — firm-level identity")
# ─────────────────────────────────────────────────────────────────────────────

run("Q7a: Registrant IDs and names matching 'artemis group'", """
    SELECT
        registrant_id,
        registrant_name,
        COUNT(DISTINCT filing_uuid) AS total_filings,
        COUNT(DISTINCT client_name) AS distinct_clients,
        ROUND(SUM(income), 0) AS total_disclosed_income,
        MIN(filing_year || '-' || filing_period) AS first_filing,
        MAX(filing_year || '-' || filing_period) AS last_filing
    FROM senate_filings
    WHERE registrant_name ILIKE '%artemis group%'
    GROUP BY registrant_id, registrant_name
    ORDER BY total_filings DESC
""")

run("Q7b: Filings naming Bridenstine — exact count and registrants", """
    SELECT
        sf.registrant_name,
        COUNT(DISTINCT sf.filing_uuid) AS filings,
        COUNT(DISTINCT sf.client_name) AS clients,
        MIN(sf.filing_year || '-' || sf.filing_period) AS first_filing,
        MAX(sf.filing_year || '-' || sf.filing_period) AS last_filing
    FROM senate_lobbyists sl
    JOIN senate_filings sf USING (filing_uuid)
    WHERE sl.lobbyist_name ILIKE '%bridenstine%'
    GROUP BY sf.registrant_name
    ORDER BY filings DESC
""")


# ─────────────────────────────────────────────────────────────────────────────
section("Q8: filing_uuids backing each NASA-targeting client claim")
# ─────────────────────────────────────────────────────────────────────────────

run("Q8a: All 24 NASA-targeting filings with client, period, and uuid", """
    SELECT
        sf.client_name,
        sf.filing_year,
        sf.filing_period,
        sf.filing_type,
        sf.filing_uuid,
        sf.income,
        sf.source_path
    FROM senate_lobbyists sl
    JOIN senate_filings sf USING (filing_uuid)
    JOIN senate_gov_entities sge USING (filing_uuid)
    WHERE sl.lobbyist_name ILIKE '%bridenstine%'
      AND sge.entity_name ILIKE '%NASA%'
    ORDER BY sf.client_name, sf.filing_year, sf.filing_period
""")


# ─────────────────────────────────────────────────────────────────────────────
section("Q9: filing_uuids for the program-overlap claims")
# ─────────────────────────────────────────────────────────────────────────────

run("Q9a: Redwire filings naming the Gateway Program", """
    SELECT DISTINCT
        sf.filing_uuid,
        sf.filing_year || '-' || sf.filing_period AS period,
        sa.general_issue_code,
        sa.description,
        sf.source_path
    FROM senate_lobbyists sl
    JOIN senate_filings sf USING (filing_uuid)
    JOIN senate_activities sa USING (filing_uuid)
    WHERE sl.lobbyist_name ILIKE '%bridenstine%'
      AND sf.client_name ILIKE '%redwire%'
      AND sa.description ILIKE '%gateway%'
    ORDER BY period
""")

run("Q9b: Lunar Outpost — confirm lunar surface mobility filing", """
    SELECT DISTINCT
        sf.filing_uuid,
        sf.filing_year || '-' || sf.filing_period AS period,
        sa.general_issue_code,
        sa.description,
        sf.source_path
    FROM senate_lobbyists sl
    JOIN senate_filings sf USING (filing_uuid)
    JOIN senate_activities sa USING (filing_uuid)
    WHERE sl.lobbyist_name ILIKE '%bridenstine%'
      AND sf.client_name ILIKE '%lunar outpost%'
    ORDER BY period
""")

run("Q9c: University of Arizona — OSIRIS-APEX filings", """
    SELECT DISTINCT
        sf.filing_uuid,
        sf.filing_year || '-' || sf.filing_period AS period,
        sa.general_issue_code,
        sa.description,
        sf.source_path
    FROM senate_lobbyists sl
    JOIN senate_filings sf USING (filing_uuid)
    JOIN senate_activities sa USING (filing_uuid)
    WHERE sl.lobbyist_name ILIKE '%bridenstine%'
      AND sf.client_name ILIKE '%arizona%'
      AND sa.description ILIKE '%osiris%'
    ORDER BY period
""")

run("Q9d: Impulse Space — NASA Authorization filings", """
    SELECT DISTINCT
        sf.filing_uuid,
        sf.filing_year || '-' || sf.filing_period AS period,
        sa.general_issue_code,
        sa.description,
        sf.source_path
    FROM senate_lobbyists sl
    JOIN senate_filings sf USING (filing_uuid)
    JOIN senate_activities sa USING (filing_uuid)
    WHERE sl.lobbyist_name ILIKE '%bridenstine%'
      AND sf.client_name ILIKE '%impulse space%'
      AND sa.description ILIKE '%NASA%'
    ORDER BY period
""")

run("Q9e: ULA — NASA Authorization Act / NDAA / appropriations", """
    SELECT DISTINCT
        sf.filing_uuid,
        sf.filing_year || '-' || sf.filing_period AS period,
        sa.general_issue_code,
        sa.description,
        sf.source_path
    FROM senate_lobbyists sl
    JOIN senate_filings sf USING (filing_uuid)
    JOIN senate_activities sa USING (filing_uuid)
    WHERE sl.lobbyist_name ILIKE '%bridenstine%'
      AND sf.client_name ILIKE '%united launch%'
      AND (sa.description ILIKE '%NASA%' OR sa.description ILIKE '%NDAA%')
    ORDER BY period
""")

run("Q9f: Voyager Space — all activity descriptions", """
    SELECT DISTINCT
        sf.filing_uuid,
        sf.filing_year || '-' || sf.filing_period AS period,
        sa.general_issue_code,
        sa.description,
        sf.source_path
    FROM senate_lobbyists sl
    JOIN senate_filings sf USING (filing_uuid)
    JOIN senate_activities sa USING (filing_uuid)
    WHERE sl.lobbyist_name ILIKE '%bridenstine%'
      AND sf.client_name ILIKE '%voyager%'
    ORDER BY period
""")


# ─────────────────────────────────────────────────────────────────────────────
section("Q10: House LDA cross-validation")
# ─────────────────────────────────────────────────────────────────────────────

run("Q10a: House filings naming Bridenstine as lobbyist", """
    SELECT
        hf.org_name AS firm,
        hf.client_name,
        hf.filing_year,
        hf.filing_period,
        hf.house_id,
        hf.senate_id,
        hf.income,
        hf.source_path
    FROM house_lobbyists hl
    JOIN house_filings hf USING (house_id)
    WHERE hl.lobbyist_name ILIKE '%bridenstine%'
    ORDER BY hf.filing_year, hf.filing_period, hf.client_name
""")

run("Q10b: House filings registered by The Artemis Group", """
    SELECT
        hf.client_name,
        hf.filing_year,
        hf.filing_period,
        hf.house_id,
        hf.senate_id,
        hf.income,
        hf.source_path
    FROM house_filings hf
    WHERE hf.org_name ILIKE '%artemis group%'
    ORDER BY hf.filing_year, hf.filing_period, hf.client_name
""")

run("Q10c: Reconciliation — Senate filings with senate_id matched in House", """
    WITH artemis_senate AS (
        SELECT DISTINCT sf.filing_uuid, sf.client_name, sf.filing_year, sf.filing_period
        FROM senate_lobbyists sl
        JOIN senate_filings sf USING (filing_uuid)
        WHERE sl.lobbyist_name ILIKE '%bridenstine%'
    )
    SELECT
        ar.client_name,
        ar.filing_year,
        ar.filing_period,
        ar.filing_uuid AS senate_uuid,
        hf.house_id,
        hf.income AS house_income
    FROM artemis_senate ar
    LEFT JOIN house_filings hf
      ON hf.senate_id IS NOT NULL
     AND (hf.org_name ILIKE '%artemis%' OR hf.client_name = ar.client_name)
     AND hf.filing_year = ar.filing_year
     AND hf.filing_period = CASE ar.filing_period
            WHEN 'first_quarter'  THEN 'Q1'
            WHEN 'second_quarter' THEN 'Q2'
            WHEN 'third_quarter'  THEN 'Q3'
            WHEN 'fourth_quarter' THEN 'Q4'
            ELSE ar.filing_period END
    ORDER BY ar.filing_year, ar.filing_period, ar.client_name
""")


# ─────────────────────────────────────────────────────────────────────────────
section("Q11: Press release corpus — Bridenstine and NASA/Artemis rhetoric")
# ─────────────────────────────────────────────────────────────────────────────

run("Q11a: Press releases mentioning Bridenstine by name (2024-2026)", """
    SELECT
        pr.date,
        pr.member_name,
        pr.party,
        pr.chamber,
        pr.title,
        pr.url
    FROM press_releases pr
    WHERE pr.text ILIKE '%bridenstine%'
      AND pr.date >= '2024-01-01'
    ORDER BY pr.date DESC
    LIMIT 50
""")

run("Q11b: Press releases mentioning Artemis Group (the firm) by name", """
    SELECT
        pr.date,
        pr.member_name,
        pr.party,
        pr.chamber,
        pr.title,
        pr.url
    FROM press_releases pr
    WHERE pr.text ILIKE '%artemis group%'
    ORDER BY pr.date DESC
""")

run("Q11c: Press releases on NASA Authorization Act / Gateway Program (2024-2026)", """
    SELECT
        pr.date,
        pr.member_name,
        pr.party,
        pr.chamber,
        pr.title,
        pr.url
    FROM press_releases pr
    WHERE (
        pr.text ILIKE '%NASA Authorization Act%'
        OR pr.text ILIKE '%Gateway Program%'
        OR pr.text ILIKE '%Commercial LEO Destinations%'
        OR pr.text ILIKE '%Commercial Lunar Payload%'
    )
    AND pr.date >= '2024-01-01'
    ORDER BY pr.date DESC
    LIMIT 30
""")

run("Q11d: Senate Commerce / House Science members talking about Starship/Artemis architecture", """
    SELECT
        pr.date,
        pr.member_name,
        pr.party,
        pr.chamber,
        pr.title,
        pr.url
    FROM press_releases pr
    WHERE (
        (pr.text ILIKE '%Starship%' AND pr.text ILIKE '%Artemis%')
        OR (pr.text ILIKE '%lunar lander%' AND pr.text ILIKE '%Artemis%')
        OR (pr.text ILIKE '%Defense Production Act%' AND pr.text ILIKE '%moon%')
    )
    AND pr.date >= '2024-01-01'
    ORDER BY pr.date DESC
    LIMIT 30
""")


# ─────────────────────────────────────────────────────────────────────────────
section("Q12: Procurement specificity — do filings name contract awards?")
# ─────────────────────────────────────────────────────────────────────────────

run("Q12a: Bridenstine filing descriptions mentioning specific contracts/awards", """
    SELECT DISTINCT
        sf.client_name,
        sf.filing_uuid,
        sf.filing_year || '-' || sf.filing_period AS period,
        sa.description
    FROM senate_lobbyists sl
    JOIN senate_filings sf USING (filing_uuid)
    JOIN senate_activities sa USING (filing_uuid)
    WHERE sl.lobbyist_name ILIKE '%bridenstine%'
      AND (
          sa.description ILIKE '%contract%'
          OR sa.description ILIKE '%procurement%'
          OR sa.description ILIKE '%award%'
          OR sa.description ILIKE '%solicitation%'
          OR sa.description ILIKE '%RFP%'
          OR sa.description ILIKE '%task order%'
      )
    ORDER BY sf.client_name, period
""")


# ─────────────────────────────────────────────────────────────────────────────
section("Q13: Income aggregation sanity check")
# ─────────────────────────────────────────────────────────────────────────────

run("Q13a: Income disclosure pattern — how many filings disclose income?", """
    SELECT
        COUNT(*) AS total_filings,
        COUNT(income) AS filings_with_income,
        ROUND(SUM(income), 0) AS total_disclosed,
        ROUND(AVG(income), 0) AS avg_disclosed,
        ROUND(MEDIAN(income), 0) AS median_disclosed,
        ROUND(MAX(income), 0) AS max_disclosed
    FROM (
        SELECT DISTINCT sf.filing_uuid, sf.income
        FROM senate_lobbyists sl
        JOIN senate_filings sf USING (filing_uuid)
        WHERE sl.lobbyist_name ILIKE '%bridenstine%'
    )
""")

con.close()
print()
print("Done. Update notes/05_finding_bridenstine.md with the filing_uuids from each query.")


Q7: The Artemis Group — firm-level identity

-- Q7a: Registrant IDs and names matching 'artemis group' --
('401108974', 'THE ARTEMIS GROUP, LLC (OKLAHOMA)', 133, 27, 4755000.0, '2024-fourth_quarter', '2026-first_quarter')

-- Q7b: Filings naming Bridenstine — exact count and registrants --
('THE ARTEMIS GROUP, LLC (OKLAHOMA)', 73, 24, '2024-fourth_quarter', '2026-first_quarter')

Q8: filing_uuids backing each NASA-targeting client claim

-- Q8a: All 24 NASA-targeting filings with client, period, and uuid --
('ALL POINTS LLC', 2025, 'first_quarter', 'Q1', '57a1f35e-01de-492a-af2c-e785eaa0a79f', 50000.0, 'data\\data\\senate\\2025\\filings\\filings_2025.json')
('ALL POINTS LLC', 2025, 'first_quarter', 'Q1', '57a1f35e-01de-492a-af2c-e785eaa0a79f', 50000.0, 'data\\data\\senate\\2025\\filings\\filings_2025.json')
('ALL POINTS LLC', 2025, 'first_quarter', 'Q1', '57a1f35e-01de-492a-af2c-e785eaa0a79f', 50000.0, 'data\\data\\senate\\2025\\filings\\filings_2025.json')
('ALL POINTS LLC', 2025, 'f

BinderException: Binder Error: No function matches the given name and argument types '~~*(INTEGER, STRING_LITERAL)'. You might need to add explicit type casts.
	Candidate functions:
	~~*(VARCHAR, VARCHAR) -> BOOLEAN


LINE 13:     WHERE hl.lobbyist_name ILIKE '%bridenstine%'
                                    ^

In [ ]:
# con = duckdb.connect("../output/investigation.duckdb", read_only=True)
run("Q8a-FIXED: Distinct NASA-targeting Bridenstine filings", """
    SELECT
        sf.client_name,
        sf.filing_year,
        sf.filing_period,
        sf.filing_uuid,
        sf.income,
        sf.source_path
    FROM senate_filings sf
    WHERE sf.filing_uuid IN (
        SELECT DISTINCT sl.filing_uuid
        FROM senate_lobbyists sl
        WHERE sl.lobbyist_name ILIKE '%bridenstine%'
    )
    AND sf.filing_uuid IN (
        SELECT DISTINCT sge.filing_uuid
        FROM senate_gov_entities sge
        WHERE sge.entity_name ILIKE '%NASA%'
    )
    ORDER BY sf.client_name, sf.filing_year, sf.filing_period
""")

run("Q8a-COUNT: Just the distinct count", """
    SELECT COUNT(DISTINCT sf.filing_uuid) AS nasa_filing_count
    FROM senate_filings sf
    WHERE sf.filing_uuid IN (
        SELECT DISTINCT sl.filing_uuid
        FROM senate_lobbyists sl
        WHERE sl.lobbyist_name ILIKE '%bridenstine%'
    )
    AND sf.filing_uuid IN (
        SELECT DISTINCT sge.filing_uuid
        FROM senate_gov_entities sge
        WHERE sge.entity_name ILIKE '%NASA%'
    )
""")

run("Q6-FIXED: Distinct gov entities targeted, no join-multiplication", """
    WITH bridenstine_filings AS (
        SELECT DISTINCT filing_uuid
        FROM senate_lobbyists
        WHERE lobbyist_name ILIKE '%bridenstine%'
    )
    SELECT
        sge.entity_name,
        COUNT(DISTINCT sge.filing_uuid) AS filings,
        COUNT(DISTINCT sf.client_name) AS num_clients,
        STRING_AGG(DISTINCT sf.client_name, ', ') AS clients
    FROM senate_gov_entities sge
    JOIN bridenstine_filings bf USING (filing_uuid)
    JOIN senate_filings sf USING (filing_uuid)
    GROUP BY sge.entity_name
    ORDER BY filings DESC
""")

con.close()


-- Q8a-FIXED: Distinct NASA-targeting Bridenstine filings --
('ALL POINTS LLC', 2025, 'first_quarter', '57a1f35e-01de-492a-af2c-e785eaa0a79f', 50000.0, 'data\\data\\senate\\2025\\filings\\filings_2025.json')
('ALL POINTS LLC', 2025, 'fourth_quarter', '1beebf33-4441-422d-9c24-205df242d613', 110000.0, 'data\\data\\senate\\2025\\filings\\filings_2025.json')
('ALL POINTS LLC', 2025, 'second_quarter', 'c60cb295-e86f-4842-b511-9dd250b567ed', 40000.0, 'data\\data\\senate\\2025\\filings\\filings_2025.json')
('ALL POINTS LLC', 2025, 'third_quarter', '51197e8e-8b33-49bb-a545-33a0e9ceb2cb', 110000.0, 'data\\data\\senate\\2025\\filings\\filings_2025.json')
('ALL POINTS LLC', 2026, 'first_quarter', 'a5ca93f6-2134-4d75-8478-93d80b2b1f0e', 110000.0, 'data\\data\\senate\\2026\\filings\\filings_2026.json')
('BLACK MOON ENERGY CORPORATION', 2025, 'first_quarter', 'c4757b60-b304-4ba5-bf74-9aefcbd47de9', 30000.0, 'data\\data\\senate\\2025\\filings\\filings_2025.json')
('EXLABS', 2025, 'third_quarter', 'd

In [3]:
run("Q14: NASA as a lobbying target — total universe and Bridenstine's share", """
    WITH all_nasa_targeting AS (
        SELECT DISTINCT filing_uuid
        FROM senate_gov_entities
        WHERE entity_name ILIKE '%NASA%'
    ),
    bridenstine_filings AS (
        SELECT DISTINCT sl.filing_uuid
        FROM senate_lobbyists sl
        WHERE sl.lobbyist_name ILIKE '%bridenstine%'
    ),
    bridenstine_nasa AS (
        SELECT b.filing_uuid
        FROM bridenstine_filings b
        JOIN all_nasa_targeting a USING (filing_uuid)
    )
    SELECT
        (SELECT COUNT(*) FROM all_nasa_targeting) AS total_nasa_targeting_filings_corpus_wide,
        (SELECT COUNT(*) FROM bridenstine_filings) AS bridenstine_total_filings,
        (SELECT COUNT(*) FROM bridenstine_nasa) AS bridenstine_nasa_filings,
        ROUND(100.0 * (SELECT COUNT(*) FROM bridenstine_nasa) / (SELECT COUNT(*) FROM all_nasa_targeting), 2) AS pct_of_nasa_lobbying_via_bridenstine
""")

run("Q15-FIXED: Top lobbyists by NASA-targeting filing count (2024-2026)", """
    SELECT
        sl.lobbyist_name,
        COUNT(DISTINCT sf.filing_uuid) AS nasa_filings,
        COUNT(DISTINCT sf.client_name) AS num_clients,
        STRING_AGG(DISTINCT sf.registrant_name, ' | ') AS firms
    FROM senate_lobbyists sl
    JOIN senate_filings sf USING (filing_uuid)
    JOIN senate_gov_entities sge USING (filing_uuid)
    WHERE sge.entity_name ILIKE '%NASA%'
      AND sl.lobbyist_name IS NOT NULL
      AND sf.filing_year >= 2024
    GROUP BY sl.lobbyist_name
    ORDER BY nasa_filings DESC
    LIMIT 20
""")

run("Q15b: NASA-targeting lobbyists with covered_position mentioning NASA", """
    SELECT
        sl.lobbyist_name,
        sl.covered_position,
        COUNT(DISTINCT sf.filing_uuid) AS nasa_filings,
        COUNT(DISTINCT sf.client_name) AS num_clients,
        STRING_AGG(DISTINCT sf.registrant_name, ' | ') AS firms
    FROM senate_lobbyists sl
    JOIN senate_filings sf USING (filing_uuid)
    JOIN senate_gov_entities sge USING (filing_uuid)
    WHERE sge.entity_name ILIKE '%NASA%'
      AND sl.lobbyist_name IS NOT NULL
      AND sf.filing_year >= 2024
      AND (sl.covered_position ILIKE '%NASA%'
           OR sl.covered_position ILIKE '%aeronautics and space%')
    GROUP BY sl.lobbyist_name, sl.covered_position
    ORDER BY nasa_filings DESC
    LIMIT 20
""")
# con.close()


-- Q14: NASA as a lobbying target — total universe and Bridenstine's share --
(1658, 73, 24, 1.45)

-- Q15-FIXED: Top lobbyists by NASA-targeting filing count (2024-2026) --
('KEVIN KELLY', 80, 11, 'ACTUM I, LLC')
('LAUREN LIPIN', 62, 9, 'ACTUM I, LLC')
('CHRISTOPHER INGRAHAM', 40, 13, 'THE ARTEMIS GROUP, LLC (OKLAHOMA)')
('JASPER THOMSON', 39, 6, 'ACTUM I, LLC')
('MARK PILAND', 36, 11, 'THE ARTEMIS GROUP, LLC (OKLAHOMA)')
('JEFFREY REGAN', 34, 5, 'ACTUM I, LLC')
('GABE SHERMAN', 32, 9, 'THE ARTEMIS GROUP, LLC (OKLAHOMA)')
('MATT TRANT', 29, 6, 'NATIONAL GROUP, LLP')
('VINCENT VERSAGE', 29, 6, 'NATIONAL GROUP, LLP')
('ALEXANDER RAUDA', 25, 5, 'ACTUM I, LLC')
('KATHRYN WALL', 24, 8, 'THE ARTEMIS GROUP, LLC (OKLAHOMA)')
('JIM BRIDENSTINE', 24, 9, 'THE ARTEMIS GROUP, LLC (OKLAHOMA)')
('MEG THOMPSON', 24, 4, 'FEDERAL SCIENCE PARTNERS LLC')
('JONATHAN GLEDHILL', 23, 3, 'POLICY NAVIGATION GROUP')
('JOHN MARTENS', 20, 3, 'FEDERAL SCIENCE PARTNERS LLC')
('JOHN CULBERSON', 19, 2, 'FEDERAL SCIE

In [5]:
run("Q16: The Artemis Group's TOTAL NASA-lobbying footprint vs. corpus", """
    WITH all_nasa_targeting AS (
        SELECT DISTINCT filing_uuid
        FROM senate_gov_entities
        WHERE entity_name ILIKE '%NASA%'
    ),
    artemis_filings AS (
        SELECT DISTINCT filing_uuid
        FROM senate_filings
        WHERE registrant_name ILIKE '%artemis group%'
    ),
    artemis_nasa AS (
        SELECT af.filing_uuid
        FROM artemis_filings af
        JOIN all_nasa_targeting ant USING (filing_uuid)
    )
    SELECT
        (SELECT COUNT(*) FROM all_nasa_targeting) AS total_nasa_filings_corpus,
        (SELECT COUNT(*) FROM artemis_filings) AS artemis_total_filings,
        (SELECT COUNT(*) FROM artemis_nasa) AS artemis_nasa_filings,
        ROUND(100.0 * (SELECT COUNT(*) FROM artemis_nasa)
              / (SELECT COUNT(*) FROM all_nasa_targeting), 2)
            AS pct_corpus_nasa_via_artemis,
        ROUND(100.0 * (SELECT COUNT(*) FROM artemis_nasa)
              / (SELECT COUNT(*) FROM artemis_filings), 2)
            AS pct_artemis_filings_targeting_nasa
""")


-- Q16: The Artemis Group's TOTAL NASA-lobbying footprint vs. corpus --
(1658, 133, 52, 3.14, 39.1)


In [6]:
run("Q17: NASA-share concentration for top NASA-targeting firms (2024-2026)", """
    WITH nasa_filings AS (
        SELECT DISTINCT filing_uuid
        FROM senate_gov_entities
        WHERE entity_name ILIKE '%NASA%'
    )
    SELECT
        sf.registrant_name AS firm,
        COUNT(DISTINCT sf.filing_uuid) AS total_filings,
        COUNT(DISTINCT CASE WHEN sf.filing_uuid IN (SELECT filing_uuid FROM nasa_filings)
                            THEN sf.filing_uuid END) AS nasa_filings,
        ROUND(100.0 *
              COUNT(DISTINCT CASE WHEN sf.filing_uuid IN (SELECT filing_uuid FROM nasa_filings)
                                  THEN sf.filing_uuid END)
              / COUNT(DISTINCT sf.filing_uuid), 1) AS pct_nasa_focus
    FROM senate_filings sf
    WHERE sf.filing_year >= 2024
    GROUP BY sf.registrant_name
    HAVING nasa_filings >= 10
    ORDER BY pct_nasa_focus DESC
    LIMIT 20
""")


-- Q17: NASA-share concentration for top NASA-targeting firms (2024-2026) --
('INFORMATION TECHNOLOGY INDUSTRY COUNCIL', 11, 11, 100.0)
('THE UNIVERSITY OF FLORIDA BOARD OF TRUSTEES', 11, 11, 100.0)
('UNITED LAUNCH ALLIANCE LLC', 15, 15, 100.0)
('ROLLS-ROYCE NORTH AMERICA AND ITS AFFILIATES', 13, 13, 100.0)
('AMERICAN ASSOCIATION OF MUSEUMS', 10, 10, 100.0)
('AEROSPACE INDUSTRIES ASSOCIATION OF AMERICA, INC.', 11, 11, 100.0)
('COMMERCIAL SPACEFLIGHT FEDERATION', 11, 11, 100.0)
('AXIOM SPACE, INC.', 10, 10, 100.0)
('MUSEUM OF SCIENCE, BOSTON', 11, 11, 100.0)
('RELATIVITY SPACE', 11, 10, 90.9)
('POLICY NAVIGATION GROUP', 42, 23, 54.8)
('IBEX2 SOLUTIONS, LLC', 30, 16, 53.3)
('STEVE PETERSEN & ASSOCIATES', 24, 11, 45.8)
('THE ARTEMIS GROUP, LLC (OKLAHOMA)', 133, 52, 39.1)
('MADISON GOVERNMENT AFFAIRS', 35, 10, 28.6)
('MS. ELIZABETH LAVACH', 66, 18, 27.3)
('FEDERAL SCIENCE PARTNERS LLC', 110, 24, 21.8)
('NATIONAL GROUP, LLP', 159, 29, 18.2)
('ACTUM I, LLC', 470, 80, 17.0)
('CORNERSTONE GOV

In [8]:
"""
05c_verification_round_2.py

Verifies the remaining open items in the Bridenstine finding:
- Ingraham, Piland, Wall: who are they, what were their prior roles
- Other Artemis Group lobbyists with prior government experience
- Press release scan for Bridenstine / Artemis Group / Starship-Artemis rhetoric
- Bridenstine's filings naming specific procurement or contract awards
"""


# ─────────────────────────────────────────────────────────────────────────────
section("Q18: All Artemis Group lobbyists, with their covered_positions")
# ─────────────────────────────────────────────────────────────────────────────

run("Q18a: Distinct lobbyists at The Artemis Group and their backgrounds", """
    SELECT
        sl.lobbyist_name,
        sl.covered_position,
        COUNT(DISTINCT sf.filing_uuid) AS filings,
        COUNT(DISTINCT sf.client_name) AS clients
    FROM senate_lobbyists sl
    JOIN senate_filings sf USING (filing_uuid)
    WHERE sf.registrant_name ILIKE '%artemis group%'
      AND sl.lobbyist_name IS NOT NULL
    GROUP BY sl.lobbyist_name, sl.covered_position
    ORDER BY filings DESC
""")


# ─────────────────────────────────────────────────────────────────────────────
section("Q19: Press release corpus — Bridenstine, Artemis, NASA programs")
# ─────────────────────────────────────────────────────────────────────────────

run("Q19a: Press releases mentioning Bridenstine by name (2024-2026)", """
    SELECT
        pr.date,
        pr.member_name,
        pr.party,
        pr.chamber,
        pr.title,
        pr.url
    FROM press_releases pr
    WHERE pr.text ILIKE '%bridenstine%'
      AND pr.date >= '2024-01-01'
    ORDER BY pr.date DESC
    LIMIT 50
""")

run("Q19b: Press releases mentioning 'Artemis Group' (the firm)", """
    SELECT
        pr.date,
        pr.member_name,
        pr.party,
        pr.chamber,
        pr.title,
        pr.url
    FROM press_releases pr
    WHERE pr.text ILIKE '%artemis group%'
    ORDER BY pr.date DESC
""")

run("Q19c: Press releases on NASA Authorization Act / Gateway Program (2024-2026)", """
    SELECT
        pr.date,
        pr.member_name,
        pr.party,
        pr.chamber,
        pr.title,
        pr.url
    FROM press_releases pr
    WHERE (
        pr.text ILIKE '%NASA Authorization Act%'
        OR pr.text ILIKE '%Gateway Program%'
        OR pr.text ILIKE '%Commercial LEO Destinations%'
        OR pr.text ILIKE '%Commercial Lunar Payload%'
    )
    AND pr.date >= '2024-01-01'
    ORDER BY pr.date DESC
    LIMIT 50
""")

run("Q19d: Press releases on Starship-vs-Artemis-architecture debate", """
    SELECT
        pr.date,
        pr.member_name,
        pr.party,
        pr.chamber,
        pr.title,
        pr.url
    FROM press_releases pr
    WHERE (
        (pr.text ILIKE '%Starship%' AND pr.text ILIKE '%Artemis%')
        OR (pr.text ILIKE '%lunar lander%' AND pr.text ILIKE '%Artemis%')
        OR (pr.text ILIKE '%human landing system%')
        OR (pr.text ILIKE '%Defense Production Act%' AND pr.text ILIKE '%moon%')
    )
    AND pr.date >= '2024-01-01'
    ORDER BY pr.date DESC
    LIMIT 50
""")


# ─────────────────────────────────────────────────────────────────────────────
section("Q20: Bridenstine filings — procurement specificity")
# ─────────────────────────────────────────────────────────────────────────────

run("Q20a: Filing descriptions mentioning specific contracts/awards/procurement", """
    SELECT DISTINCT
        sf.client_name,
        sf.filing_uuid,
        sf.filing_year || '-' || sf.filing_period AS period,
        sa.description
    FROM senate_lobbyists sl
    JOIN senate_filings sf USING (filing_uuid)
    JOIN senate_activities sa USING (filing_uuid)
    WHERE sf.registrant_name ILIKE '%artemis group%'
      AND (
          sa.description ILIKE '%contract%'
          OR sa.description ILIKE '%procurement%'
          OR sa.description ILIKE '%award%'
          OR sa.description ILIKE '%solicitation%'
          OR sa.description ILIKE '%RFP%'
          OR sa.description ILIKE '%task order%'
          OR sa.description ILIKE '%selection%'
      )
    ORDER BY sf.client_name, period
""")


# ─────────────────────────────────────────────────────────────────────────────
section("Q21: Members of Congress whose press releases mention named Artemis clients")
# ─────────────────────────────────────────────────────────────────────────────
# This is the say-vs-pay angle — do members publicly champion companies
# that are lobbying them via Bridenstine's firm?

run("Q21a: Press releases mentioning ULA / Voyager / Redwire / Lunar Outpost (2024-2026)", """
    SELECT
        pr.date,
        pr.member_name,
        pr.party,
        pr.chamber,
        pr.title,
        pr.url,
        CASE
            WHEN pr.text ILIKE '%United Launch Alliance%' OR pr.text ILIKE '%ULA%' THEN 'ULA'
            WHEN pr.text ILIKE '%Voyager Space%'                                    THEN 'Voyager'
            WHEN pr.text ILIKE '%Redwire%'                                          THEN 'Redwire'
            WHEN pr.text ILIKE '%Lunar Outpost%'                                    THEN 'Lunar Outpost'
            WHEN pr.text ILIKE '%Impulse Space%'                                    THEN 'Impulse Space'
            WHEN pr.text ILIKE '%Quantum Space%'                                    THEN 'Quantum Space'
            ELSE 'other'
        END AS named_client
    FROM press_releases pr
    WHERE (
        pr.text ILIKE '%United Launch Alliance%'
        OR pr.text ILIKE '% ULA %'
        OR pr.text ILIKE '%Voyager Space%'
        OR pr.text ILIKE '%Redwire%'
        OR pr.text ILIKE '%Lunar Outpost%'
        OR pr.text ILIKE '%Impulse Space%'
        OR pr.text ILIKE '%Quantum Space%'
    )
    AND pr.date >= '2024-01-01'
    ORDER BY named_client, pr.date DESC
    LIMIT 100
""")


# ─────────────────────────────────────────────────────────────────────────────
section("Q22: Sherman's filings — what does he target?")
# ─────────────────────────────────────────────────────────────────────────────

run("Q22a: Gov entities targeted in Sherman's filings", """
    WITH sherman_filings AS (
        SELECT DISTINCT filing_uuid
        FROM senate_lobbyists
        WHERE lobbyist_name ILIKE '%gabe sherman%'
           OR lobbyist_name ILIKE '%gabriel sherman%'
    )
    SELECT
        sge.entity_name,
        COUNT(DISTINCT sge.filing_uuid) AS filings,
        COUNT(DISTINCT sf.client_name) AS num_clients
    FROM senate_gov_entities sge
    JOIN sherman_filings sf_filt USING (filing_uuid)
    JOIN senate_filings sf USING (filing_uuid)
    GROUP BY sge.entity_name
    ORDER BY filings DESC
""")

run("Q22b: Sherman's client list", """
    SELECT
        sf.client_name,
        COUNT(DISTINCT sf.filing_uuid) AS filings,
        ROUND(SUM(sf.income), 0) AS total_disclosed,
        MIN(sf.filing_year || '-' || sf.filing_period) AS first,
        MAX(sf.filing_year || '-' || sf.filing_period) AS last
    FROM senate_lobbyists sl
    JOIN senate_filings sf USING (filing_uuid)
    WHERE sl.lobbyist_name ILIKE '%gabe sherman%'
       OR sl.lobbyist_name ILIKE '%gabriel sherman%'
    GROUP BY sf.client_name
    ORDER BY filings DESC
""")

# con.close()


Q18: All Artemis Group lobbyists, with their covered_positions

-- Q18a: Distinct lobbyists at The Artemis Group and their backgrounds --
('GABE SHERMAN', 'District Director, United States House of Representatives, Hon. Bridenstine; Chief of Staff, NASA', 80, 24)
('JIM BRIDENSTINE', 'Member, United States House of Representatives; Administrator, NASA', 73, 24)
('MARK PILAND', 'Chief of Staff - Rep. Ralph Norman; Chief of Staff/Leg. Director/Senior Leg. Asst./Leg. Correspondent - Rep. Jim Bridenstine; Intern - Rep. Raul Labrador', 70, 21)
('CHRISTOPHER INGRAHAM', 'Professional staff - House Space Subcommittee; Senior Policy Advisor - Rep. Jim Bridenstine; LA - Rep. Trey Gowdy; intern - Sen. Jim DeMint', 69, 22)
('KATHRYN WALL', 'Chief of Staff, National Space Council; Deputy Director of Scheduling, Office of the Vice President', 46, 16)
('JIM BRIDENSTINE', None, 15, 4)
('ANDREW HEVENER', None, 14, 9)
('GABE SHERMAN', None, 10, 2)
('SHAWN BARNES', 'Deputy Assistant Secretary of the Air 